In [4]:
from pathlib import Path
import pandas as pd

# ------------------------------------------------------------
# Step 1: Time-based split on execution-level dataset
# ------------------------------------------------------------

cwd = Path.cwd()

if (cwd / "subset").exists():
    base_dir = cwd / "subset"
elif (cwd.parent / "subset").exists():
    base_dir = cwd.parent / "subset"
elif cwd.name == "subset":
    base_dir = cwd
else:
    raise FileNotFoundError(f"Could not locate 'subset' folder from cwd={cwd}")

INPUT_PATH = base_dir / "job_metrics_v0.parquet"
OUT_TRAIN = base_dir / "job_metrics_train_v0.parquet"
OUT_VAL   = base_dir / "job_metrics_val_v0.parquet"
OUT_TEST  = base_dir / "job_metrics_test_v0.parquet"

TRAIN_FRAC = 0.70
VAL_FRAC = 0.15
TEST_FRAC = 0.15

# safety check
assert abs((TRAIN_FRAC + VAL_FRAC + TEST_FRAC) - 1.0) < 1e-9, "Split fractions must sum to 1.0"

# ------------------------------------------------------------
# Load
# ------------------------------------------------------------
print("Current working directory:", cwd)
print("Using input:", INPUT_PATH.resolve())

df = pd.read_parquet(INPUT_PATH)
print(f"Loaded rows: {len(df):,}")
print("Columns:", list(df.columns))

# ------------------------------------------------------------
# Parse and validate time column
# ------------------------------------------------------------
if "start_time" not in df.columns:
    raise ValueError("Expected 'start_time' column not found.")

df["start_time"] = pd.to_datetime(df["start_time"], errors="coerce")
df["end_time"] = pd.to_datetime(df["end_time"], errors="coerce") if "end_time" in df.columns else pd.NaT

bad_start = df["start_time"].isna().sum()
if bad_start > 0:
    print(f"Dropping {bad_start:,} rows with invalid start_time")
    df = df.dropna(subset=["start_time"]).copy()

# ------------------------------------------------------------
# Sort by time
# ------------------------------------------------------------
df = df.sort_values(["start_time", "execution_id"], kind="mergesort").reset_index(drop=True)

# ------------------------------------------------------------
# Split by time order
# ------------------------------------------------------------
n = len(df)
train_end = int(n * TRAIN_FRAC)
val_end = int(n * (TRAIN_FRAC + VAL_FRAC))

train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------
train_df.to_parquet(OUT_TRAIN, index=False)
val_df.to_parquet(OUT_VAL, index=False)
test_df.to_parquet(OUT_TEST, index=False)

# ------------------------------------------------------------
# Validation checks
# ------------------------------------------------------------
print("\nSaved files:")
print(OUT_TRAIN.resolve())
print(OUT_VAL.resolve())
print(OUT_TEST.resolve())

print("\nRow counts:")
print(f"Train: {len(train_df):,}")
print(f"Val  : {len(val_df):,}")
print(f"Test : {len(test_df):,}")
print(f"Total: {len(train_df) + len(val_df) + len(test_df):,}")

assert len(train_df) + len(val_df) + len(test_df) == len(df), "Row counts do not add up."

def time_range(name, part):
    if len(part) == 0:
        print(f"{name}: EMPTY")
        return
    print(
        f"{name}: "
        f"{part['start_time'].min()}  -->  {part['start_time'].max()}"
    )

print("\nTime ranges:")
time_range("Train", train_df)
time_range("Val  ", val_df)
time_range("Test ", test_df)

if len(train_df) > 0 and len(val_df) > 0:
    assert train_df["start_time"].max() <= val_df["start_time"].min(), \
        "Train/Val temporal ordering is broken."

if len(val_df) > 0 and len(test_df) > 0:
    assert val_df["start_time"].max() <= test_df["start_time"].min(), \
        "Val/Test temporal ordering is broken."

print("\nSplit completed successfully.")

Current working directory: /Users/meghanakoti/masterproject/masters-project/notebooks
Using input: /Users/meghanakoti/masterproject/masters-project/subset/job_metrics_v0.parquet
Loaded rows: 98,651
Columns: ['execution_id', 'recurring_job_id', 'recurrence_key', 'start_time', 'end_time', 'duration', 'n_history', 'run_count', 'confidence_tier', 'req_cpu', 'req_mem', 'peak_cpu', 'peak_mem', 'avg_cpu', 'avg_mem', 'slack_cpu', 'slack_mem', 'viol_cpu', 'viol_mem', 'viol_any']

Saved files:
/Users/meghanakoti/masterproject/masters-project/subset/job_metrics_train_v0.parquet
/Users/meghanakoti/masterproject/masters-project/subset/job_metrics_val_v0.parquet
/Users/meghanakoti/masterproject/masters-project/subset/job_metrics_test_v0.parquet

Row counts:
Train: 69,055
Val  : 14,798
Test : 14,798
Total: 98,651

Time ranges:
Train: 1970-01-01 00:00:00.600000  -->  1970-01-01 00:30:12.600000
Val  : 1970-01-01 00:30:12.600000  -->  1970-01-01 00:37:29.100000
Test : 1970-01-01 00:37:29.100000  -->  19

In [6]:
from pathlib import Path
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Step 2: Build train-only job profiles
# ------------------------------------------------------------

cwd = Path.cwd()

if (cwd / "subset").exists():
    base_dir = cwd / "subset"
elif (cwd.parent / "subset").exists():
    base_dir = cwd.parent / "subset"
elif cwd.name == "subset":
    base_dir = cwd
else:
    raise FileNotFoundError(f"Could not locate 'subset' folder from cwd={cwd}")

TRAIN_PATH = base_dir / "job_metrics_train_v0.parquet"
OUT_PROFILES_TRAIN = base_dir / "job_profiles_train_only_v0.parquet"

print("Current working directory:", cwd)
print("Using train input:", TRAIN_PATH.resolve())

df = pd.read_parquet(TRAIN_PATH)

print(f"Loaded train rows: {len(df):,}")
print("Columns:", list(df.columns))

# ------------------------------------------------------------
# Basic cleanup
# ------------------------------------------------------------
required_cols = [
    "recurrence_key",
    "peak_cpu",
    "peak_mem",
    "duration",
]

missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

work = df[required_cols].copy()

# Drop rows without grouping key
before = len(work)
work = work.dropna(subset=["recurrence_key"])
print(f"Dropped rows with null recurrence_key: {before - len(work):,}")

# Ensure numeric
for col in ["peak_cpu", "peak_mem", "duration"]:
    work[col] = pd.to_numeric(work[col], errors="coerce")

# Optional: drop rows with all metrics missing
before = len(work)
work = work.dropna(subset=["peak_cpu", "peak_mem", "duration"], how="all")
print(f"Dropped rows with all key metrics missing: {before - len(work):,}")

# ------------------------------------------------------------
# Aggregate train-only profiles
# ------------------------------------------------------------
profiles = (
    work.groupby("recurrence_key", dropna=False)
    .agg(
        run_count=("recurrence_key", "size"),
        mean_peak_cpu=("peak_cpu", "mean"),
        std_peak_cpu=("peak_cpu", "std"),
        mean_peak_mem=("peak_mem", "mean"),
        std_peak_mem=("peak_mem", "std"),
        mean_duration=("duration", "mean"),
        p90_cpu=("peak_cpu", lambda s: s.quantile(0.90)),
        p95_cpu=("peak_cpu", lambda s: s.quantile(0.95)),
        p99_cpu=("peak_cpu", lambda s: s.quantile(0.99)),
        p90_mem=("peak_mem", lambda s: s.quantile(0.90)),
        p95_mem=("peak_mem", lambda s: s.quantile(0.95)),
        p99_mem=("peak_mem", lambda s: s.quantile(0.99)),
    )
    .reset_index()
)

# ------------------------------------------------------------
# Derived stability features
# ------------------------------------------------------------
profiles["std_peak_cpu"] = profiles["std_peak_cpu"].fillna(0.0)
profiles["std_peak_mem"] = profiles["std_peak_mem"].fillna(0.0)

profiles["cv_cpu"] = np.where(
    profiles["mean_peak_cpu"].abs() > 1e-12,
    profiles["std_peak_cpu"] / profiles["mean_peak_cpu"],
    np.nan
)

profiles["cv_mem"] = np.where(
    profiles["mean_peak_mem"].abs() > 1e-12,
    profiles["std_peak_mem"] / profiles["mean_peak_mem"],
    np.nan
)

profiles["stability"] = profiles[["cv_cpu", "cv_mem"]].mean(axis=1, skipna=True)

profiles["is_recurring"] = profiles["run_count"] >= 2

# ------------------------------------------------------------
# Confidence tier
# ------------------------------------------------------------
def assign_confidence_tier(run_count):
    if pd.isna(run_count):
        return "low"
    if run_count >= 20:
        return "high"
    elif run_count >= 5:
        return "medium"
    else:
        return "low"

profiles["confidence_tier"] = profiles["run_count"].apply(assign_confidence_tier)

# ------------------------------------------------------------
# Optional descriptive labels
# ------------------------------------------------------------
def resource_profile(row):
    cpu = row["mean_peak_cpu"]
    mem = row["mean_peak_mem"]
    if pd.isna(cpu) or pd.isna(mem):
        return "unknown"
    if cpu >= mem:
        return "cpu_heavy"
    return "mem_heavy"

def duration_profile(d):
    if pd.isna(d):
        return "unknown"
    if d < 60:
        return "short"
    elif d < 300:
        return "medium"
    return "long"

def recurrence_profile(rc):
    if pd.isna(rc):
        return "unknown"
    if rc == 1:
        return "one_off"
    elif rc < 5:
        return "low_repeat"
    elif rc < 20:
        return "repeat"
    return "high_repeat"

def stability_profile(s):
    if pd.isna(s):
        return "unknown"
    if s < 0.10:
        return "stable"
    elif s < 0.30:
        return "moderate"
    return "volatile"

profiles["resource_profile"] = profiles.apply(resource_profile, axis=1)
profiles["duration_profile"] = profiles["mean_duration"].apply(duration_profile)
profiles["recurrence_profile"] = profiles["run_count"].apply(recurrence_profile)
profiles["stability_profile"] = profiles["stability"].apply(stability_profile)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------
profile_cols = [
    "recurrence_key",
    "run_count",
    "mean_peak_cpu",
    "std_peak_cpu",
    "mean_peak_mem",
    "std_peak_mem",
    "mean_duration",
    "cv_cpu",
    "cv_mem",
    "stability",
    "is_recurring",
    "confidence_tier",
    "resource_profile",
    "duration_profile",
    "recurrence_profile",
    "stability_profile",
    "p90_cpu",
    "p95_cpu",
    "p99_cpu",
    "p90_mem",
    "p95_mem",
    "p99_mem",
]

profiles = profiles[profile_cols].copy()
profiles.to_parquet(OUT_PROFILES_TRAIN, index=False)

print(f"\nSaved train-only profiles: {OUT_PROFILES_TRAIN.resolve()}")
print(f"Profile rows: {len(profiles):,}")
print("Profile columns:")
print(list(profiles.columns))

print("\nConfidence tier counts:")
print(profiles["confidence_tier"].value_counts(dropna=False))

print("\nRun count summary:")
print(profiles["run_count"].describe())

Current working directory: /Users/meghanakoti/masterproject/masters-project/notebooks
Using train input: /Users/meghanakoti/masterproject/masters-project/subset/job_metrics_train_v0.parquet
Loaded train rows: 69,055
Columns: ['execution_id', 'recurring_job_id', 'recurrence_key', 'start_time', 'end_time', 'duration', 'n_history', 'run_count', 'confidence_tier', 'req_cpu', 'req_mem', 'peak_cpu', 'peak_mem', 'avg_cpu', 'avg_mem', 'slack_cpu', 'slack_mem', 'viol_cpu', 'viol_mem', 'viol_any']
Dropped rows with null recurrence_key: 0
Dropped rows with all key metrics missing: 0

Saved train-only profiles: /Users/meghanakoti/masterproject/masters-project/subset/job_profiles_train_only_v0.parquet
Profile rows: 1,143
Profile columns:
['recurrence_key', 'run_count', 'mean_peak_cpu', 'std_peak_cpu', 'mean_peak_mem', 'std_peak_mem', 'mean_duration', 'cv_cpu', 'cv_mem', 'stability', 'is_recurring', 'confidence_tier', 'resource_profile', 'duration_profile', 'recurrence_profile', 'stability_profile',

In [8]:
from pathlib import Path
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Step 3: Attach train-only profiles and generate baselines
# ------------------------------------------------------------

cwd = Path.cwd()

if (cwd / "subset").exists():
    base_dir = cwd / "subset"
elif (cwd.parent / "subset").exists():
    base_dir = cwd.parent / "subset"
elif cwd.name == "subset":
    base_dir = cwd
else:
    raise FileNotFoundError(f"Could not locate 'subset' folder from cwd={cwd}")

VAL_PATH = base_dir / "job_metrics_val_v0.parquet"
TEST_PATH = base_dir / "job_metrics_test_v0.parquet"
PROFILE_PATH = base_dir / "job_profiles_train_only_v0.parquet"

OUT_VAL_BASELINES = base_dir / "job_metrics_val_with_baselines_v0.parquet"
OUT_TEST_BASELINES = base_dir / "job_metrics_test_with_baselines_v0.parquet"

# ------------------------------------------------------------
# Load
# ------------------------------------------------------------
print("Current working directory:", cwd)
print("Using validation input:", VAL_PATH.resolve())
print("Using test input      :", TEST_PATH.resolve())
print("Using profile input   :", PROFILE_PATH.resolve())

val_df = pd.read_parquet(VAL_PATH)
test_df = pd.read_parquet(TEST_PATH)
profiles = pd.read_parquet(PROFILE_PATH)

print(f"Loaded val rows   : {len(val_df):,}")
print(f"Loaded test rows  : {len(test_df):,}")
print(f"Loaded profiles   : {len(profiles):,}")

# ------------------------------------------------------------
# Keep only needed profile columns for join
# ------------------------------------------------------------
profile_cols = [
    "recurrence_key",
    "run_count",
    "mean_peak_cpu",
    "std_peak_cpu",
    "mean_peak_mem",
    "std_peak_mem",
    "mean_duration",
    "cv_cpu",
    "cv_mem",
    "stability",
    "is_recurring",
    "confidence_tier",
    "resource_profile",
    "duration_profile",
    "recurrence_profile",
    "stability_profile",
    "p90_cpu",
    "p95_cpu",
    "p99_cpu",
    "p90_mem",
    "p95_mem",
    "p99_mem",
]

profiles = profiles[profile_cols].copy()

# Rename joined profile columns so they are clearly train-derived
rename_map = {
    "run_count": "hist_run_count",
    "mean_peak_cpu": "hist_mean_peak_cpu",
    "std_peak_cpu": "hist_std_peak_cpu",
    "mean_peak_mem": "hist_mean_peak_mem",
    "std_peak_mem": "hist_std_peak_mem",
    "mean_duration": "hist_mean_duration",
    "cv_cpu": "hist_cv_cpu",
    "cv_mem": "hist_cv_mem",
    "stability": "hist_stability",
    "is_recurring": "hist_is_recurring",
    "confidence_tier": "hist_confidence_tier",
    "resource_profile": "hist_resource_profile",
    "duration_profile": "hist_duration_profile",
    "recurrence_profile": "hist_recurrence_profile",
    "stability_profile": "hist_stability_profile",
    "p90_cpu": "hist_p90_cpu",
    "p95_cpu": "hist_p95_cpu",
    "p99_cpu": "hist_p99_cpu",
    "p90_mem": "hist_p90_mem",
    "p95_mem": "hist_p95_mem",
    "p99_mem": "hist_p99_mem",
}
profiles = profiles.rename(columns=rename_map)

# ------------------------------------------------------------
# Attach train-only profiles
# ------------------------------------------------------------
val = val_df.merge(profiles, on="recurrence_key", how="left")
test = test_df.merge(profiles, on="recurrence_key", how="left")

# ------------------------------------------------------------
# Cold-start visibility
# ------------------------------------------------------------
val_cold = val["hist_run_count"].isna().sum()
test_cold = test["hist_run_count"].isna().sum()

print("\nRows with no matching train profile (cold start / unseen recurrence_key):")
print(f"Validation: {val_cold:,}")
print(f"Test      : {test_cold:,}")

# ------------------------------------------------------------
# Generate baseline recommendation columns
# ------------------------------------------------------------
# Strategy:
# - requested baseline: original requested resources
# - mean / p90 / p95 / p99 baselines from train-only profiles
# - fallback for unseen groups: use request
def add_baselines(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    # Requested baseline
    out["base_req_cpu"] = out["req_cpu"]
    out["base_req_mem"] = out["req_mem"]

    # Mean history baseline
    out["base_mean_cpu"] = out["hist_mean_peak_cpu"].fillna(out["req_cpu"])
    out["base_mean_mem"] = out["hist_mean_peak_mem"].fillna(out["req_mem"])

    # Quantile baselines
    out["base_p90_cpu"] = out["hist_p90_cpu"].fillna(out["req_cpu"])
    out["base_p90_mem"] = out["hist_p90_mem"].fillna(out["req_mem"])

    out["base_p95_cpu"] = out["hist_p95_cpu"].fillna(out["req_cpu"])
    out["base_p95_mem"] = out["hist_p95_mem"].fillna(out["req_mem"])

    out["base_p99_cpu"] = out["hist_p99_cpu"].fillna(out["req_cpu"])
    out["base_p99_mem"] = out["hist_p99_mem"].fillna(out["req_mem"])

    # Baseline source tags
    out["base_mean_source"] = np.where(out["hist_mean_peak_cpu"].notna(), "history", "request_fallback")
    out["base_p90_source"] = np.where(out["hist_p90_cpu"].notna(), "history", "request_fallback")
    out["base_p95_source"] = np.where(out["hist_p95_cpu"].notna(), "history", "request_fallback")
    out["base_p99_source"] = np.where(out["hist_p99_cpu"].notna(), "history", "request_fallback")

    return out

val = add_baselines(val)
test = add_baselines(test)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------
val.to_parquet(OUT_VAL_BASELINES, index=False)
test.to_parquet(OUT_TEST_BASELINES, index=False)

print("\nSaved files:")
print(OUT_VAL_BASELINES.resolve())
print(OUT_TEST_BASELINES.resolve())

print("\nValidation baseline columns created:")
print([
    "base_req_cpu", "base_req_mem",
    "base_mean_cpu", "base_mean_mem",
    "base_p90_cpu", "base_p90_mem",
    "base_p95_cpu", "base_p95_mem",
    "base_p99_cpu", "base_p99_mem"
])

print("\nSample baseline coverage:")
print("Validation history-backed P95 CPU rows:", int((val["base_p95_source"] == "history").sum()))
print("Validation fallback P95 CPU rows     :", int((val["base_p95_source"] == "request_fallback").sum()))
print("Test history-backed P95 CPU rows      :", int((test["base_p95_source"] == "history").sum()))
print("Test fallback P95 CPU rows             :", int((test["base_p95_source"] == "request_fallback").sum()))

print("\nStep 3 completed: profiles attached and baselines generated.")

Current working directory: /Users/meghanakoti/masterproject/masters-project/notebooks
Using validation input: /Users/meghanakoti/masterproject/masters-project/subset/job_metrics_val_v0.parquet
Using test input      : /Users/meghanakoti/masterproject/masters-project/subset/job_metrics_test_v0.parquet
Using profile input   : /Users/meghanakoti/masterproject/masters-project/subset/job_profiles_train_only_v0.parquet
Loaded val rows   : 14,798
Loaded test rows  : 14,798
Loaded profiles   : 1,143

Rows with no matching train profile (cold start / unseen recurrence_key):
Validation: 2,694
Test      : 4,156

Saved files:
/Users/meghanakoti/masterproject/masters-project/subset/job_metrics_val_with_baselines_v0.parquet
/Users/meghanakoti/masterproject/masters-project/subset/job_metrics_test_with_baselines_v0.parquet

Validation baseline columns created:
['base_req_cpu', 'base_req_mem', 'base_mean_cpu', 'base_mean_mem', 'base_p90_cpu', 'base_p90_mem', 'base_p95_cpu', 'base_p95_mem', 'base_p99_cpu

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Step 4: Evaluate baselines on validation and test
# ------------------------------------------------------------

cwd = Path.cwd()

if (cwd / "subset").exists():
    base_dir = cwd / "subset"
elif (cwd.parent / "subset").exists():
    base_dir = cwd.parent / "subset"
elif cwd.name == "subset":
    base_dir = cwd
else:
    raise FileNotFoundError(f"Could not locate 'subset' folder from cwd={cwd}")

VAL_BASE_PATH = base_dir / "job_metrics_val_with_baselines_v0.parquet"
TEST_BASE_PATH = base_dir / "job_metrics_test_with_baselines_v0.parquet"

OUT_VAL_EVAL = base_dir / "job_metrics_val_evaluated_v2.parquet"
OUT_TEST_EVAL = base_dir / "job_metrics_test_evaluated_v2.parquet"

OUT_VAL_SUMMARY = base_dir / "validation_baseline_summary_v2.csv"
OUT_TEST_SUMMARY = base_dir / "test_baseline_summary_v2.csv"

# ------------------------------------------------------------
# Load
# ------------------------------------------------------------
print("Current working directory:", cwd)
print("Using validation baseline input:", VAL_BASE_PATH.resolve())
print("Using test baseline input      :", TEST_BASE_PATH.resolve())

val = pd.read_parquet(VAL_BASE_PATH)
test = pd.read_parquet(TEST_BASE_PATH)

print(f"Loaded validation rows: {len(val):,}")
print(f"Loaded test rows      : {len(test):,}")

# ------------------------------------------------------------
# Baselines to evaluate
# ------------------------------------------------------------
baseline_names = ["req", "mean", "p90", "p95", "p99"]

for b in baseline_names:
    for col in [f"base_{b}_cpu", f"base_{b}_mem"]:
        if col not in val.columns or col not in test.columns:
            raise ValueError(f"Missing baseline column: {col}")

required_actual_cols = ["peak_cpu", "peak_mem", "req_cpu", "req_mem"]
for col in required_actual_cols:
    if col not in val.columns or col not in test.columns:
        raise ValueError(f"Missing required actual column: {col}")

# ------------------------------------------------------------
# Utilization safeguard
# ------------------------------------------------------------
MIN_RESOURCE = 1e-3

# ------------------------------------------------------------
# Evaluation function
# ------------------------------------------------------------
def evaluate_baselines(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    for b in baseline_names:
        cpu_col = f"base_{b}_cpu"
        mem_col = f"base_{b}_mem"

        # violation flags
        out[f"viol_cpu_{b}"] = out["peak_cpu"] > out[cpu_col]
        out[f"viol_mem_{b}"] = out["peak_mem"] > out[mem_col]
        out[f"viol_any_{b}"] = out[f"viol_cpu_{b}"] | out[f"viol_mem_{b}"]

        # slack: positive = over-provisioning, negative = under-provisioning
        out[f"slack_cpu_{b}"] = out[cpu_col] - out["peak_cpu"]
        out[f"slack_mem_{b}"] = out[mem_col] - out["peak_mem"]

        # waste: over-provisioned amount only
        out[f"waste_cpu_{b}"] = np.maximum(out[f"slack_cpu_{b}"], 0)
        out[f"waste_mem_{b}"] = np.maximum(out[f"slack_mem_{b}"], 0)

        # shortfall: under-provisioned amount only
        out[f"short_cpu_{b}"] = np.maximum(out["peak_cpu"] - out[cpu_col], 0)
        out[f"short_mem_{b}"] = np.maximum(out["peak_mem"] - out[mem_col], 0)

        # utilization: actual / recommended
        out[f"cpu_util_{b}"] = np.where(
            out[cpu_col] > MIN_RESOURCE,
            out["peak_cpu"] / out[cpu_col],
            np.nan
        )
        out[f"mem_util_{b}"] = np.where(
            out[mem_col] > MIN_RESOURCE,
            out["peak_mem"] / out[mem_col],
            np.nan
        )

    return out

val_eval = evaluate_baselines(val)
test_eval = evaluate_baselines(test)

# ------------------------------------------------------------
# Build summary table
# ------------------------------------------------------------
def make_summary(df: pd.DataFrame, split_name: str) -> pd.DataFrame:
    rows = []

    for b in baseline_names:
        row = {
            "split": split_name,
            "baseline": b,
            "cpu_violation_rate": df[f"viol_cpu_{b}"].mean(),
            "mem_violation_rate": df[f"viol_mem_{b}"].mean(),
            "any_violation_rate": df[f"viol_any_{b}"].mean(),
            "avg_cpu_slack": df[f"slack_cpu_{b}"].mean(),
            "avg_mem_slack": df[f"slack_mem_{b}"].mean(),
            "avg_cpu_waste": df[f"waste_cpu_{b}"].mean(),
            "avg_mem_waste": df[f"waste_mem_{b}"].mean(),
            "avg_cpu_shortfall": df[f"short_cpu_{b}"].mean(),
            "avg_mem_shortfall": df[f"short_mem_{b}"].mean(),
            "avg_cpu_utilization": df[f"cpu_util_{b}"].mean(),
            "avg_mem_utilization": df[f"mem_util_{b}"].mean(),
            "median_cpu_utilization": df[f"cpu_util_{b}"].median(),
            "median_mem_utilization": df[f"mem_util_{b}"].median(),
            "cpu_util_valid_rows": df[f"cpu_util_{b}"].notna().sum(),
            "mem_util_valid_rows": df[f"mem_util_{b}"].notna().sum(),
        }
        rows.append(row)

    return pd.DataFrame(rows)

val_summary = make_summary(val_eval, "validation")
test_summary = make_summary(test_eval, "test")

# ------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------
val_eval.to_parquet(OUT_VAL_EVAL, index=False)
test_eval.to_parquet(OUT_TEST_EVAL, index=False)
val_summary.to_csv(OUT_VAL_SUMMARY, index=False)
test_summary.to_csv(OUT_TEST_SUMMARY, index=False)

# ------------------------------------------------------------
# Display summaries
# ------------------------------------------------------------
pd.set_option("display.float_format", lambda x: f"{x:0.4f}")

print("\nValidation summary:")
print(val_summary)

print("\nTest summary:")
print(test_summary)

print("\nSaved files:")
print(OUT_VAL_EVAL.resolve())
print(OUT_TEST_EVAL.resolve())
print(OUT_VAL_SUMMARY.resolve())
print(OUT_TEST_SUMMARY.resolve())

# ------------------------------------------------------------
# Compact comparison
# ------------------------------------------------------------
compact_cols = [
    "split",
    "baseline",
    "cpu_violation_rate",
    "mem_violation_rate",
    "any_violation_rate",
    "avg_cpu_waste",
    "avg_mem_waste",
    "avg_cpu_utilization",
    "avg_mem_utilization",
    "cpu_util_valid_rows",
    "mem_util_valid_rows",
]

compact = pd.concat([val_summary[compact_cols], test_summary[compact_cols]], ignore_index=True)

print("\nCompact comparison:")
print(compact.sort_values(["split", "baseline"]).reset_index(drop=True))